# 🔬 Logistic Regression from Scratch

**Dataset:** Breast Cancer Wisconsin (sklearn built-in) — 569 samples, 30 features  
**Task:** Binary classification — Malignant (0) vs Benign (1)  
**Method:** Logistic Regression with Sigmoid + BCE Loss + L2 Regularization  

## The Model

$$z = X \cdot W + b$$
$$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

## Binary Cross-Entropy Loss + L2 Regularization

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i + \varepsilon) + (1 - y_i) \log(1 - \hat{y}_i + \varepsilon) \right] + \frac{\lambda}{2N} \sum W^2$$

The $\varepsilon = 10^{-7}$ prevents $\log(0) \to -\infty$ (NaN catastrophe).  
The L2 term penalizes large weights, reducing overfitting.

## Gradients

$$\frac{\partial \mathcal{L}}{\partial W} = \frac{1}{N} X^T (\hat{y} - y) + \frac{\lambda}{N} W$$

$$\frac{\partial \mathcal{L}}{\partial b} = \frac{1}{N} \sum (\hat{y} - y)$$

Note: bias $b$ is **not** regularized — standard practice.

In [ ]:
# ── Imports & Config ──────────────────────────────────────────────────────────
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import yaml
from sklearn.datasets import load_breast_cancer
from src.logistic_regression import LogisticRegression

with open("../configs/logistic_config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("Config loaded:")
for section, values in config.items():
    print(f"  {section}: {values}")

In [ ]:
# ── Load & Prepare Data ───────────────────────────────────────────────────────
data = load_breast_cancer()
X    = data.data     # (569, 30) — 30 tumour measurement features
y    = data.target   # (569,)    — 0=malignant, 1=benign

# Shuffle with fixed seed
# Why shuffle: sklearn loads classes in order (all 0s then all 1s).
# Without shuffle, train set would be mostly class 0, test set mostly class 1.
seed = config["reproducibility"]["random_seed"]
rng  = np.random.default_rng(seed)
idx  = rng.permutation(len(X))
X, y = X[idx], y[idx]

# Train / test split — raw data, NO standardization here.
# Standardization happens inside LogisticRegression.fit()
# to prevent data leakage (test stats must never touch train scaler).
split   = int((1 - config["data"]["test_size"]) * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Dataset   : {data.DESCR.splitlines()[0]}")
print(f"X_train   : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_test    : {X_test.shape}    y_test  : {y_test.shape}")
print(f"Class balance (train) — 0: {(y_train==0).sum()}  1: {(y_train==1).sum()}")

In [ ]:
# ── Train Model ───────────────────────────────────────────────────────────────
model = LogisticRegression(config)
model.fit(X_train, y_train)

print(f"\nFinal loss    : {model.loss_history[-1]:.6f}")
print(f"Weights shape : {model.weights.shape}")
print(f"Bias          : {model.bias:.6f}")

In [ ]:
# ── Loss Curve ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(model.loss_history, color="#1A6BC4", linewidth=1.8, label="BCE + L2 Loss")
ax.set_xlabel("Epoch",     fontsize=12)
ax.set_ylabel("Loss",      fontsize=12)
ax.set_title("Logistic Regression — Training Loss Curve", fontsize=14)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
os.makedirs("../assets", exist_ok=True)
plt.savefig("../assets/loss_curve_logistic.png", dpi=150)
plt.show()
print("Saved → assets/loss_curve_logistic.png")

In [ ]:
# ── Evaluate on Test Set ──────────────────────────────────────────────────────
metrics = model.evaluate(X_test, y_test)

print("─" * 38)
print("  Evaluation — Test Set")
print("─" * 38)
for k, v in metrics.items():
    print(f"  {k:<18} : {v:.4f}")
print("─" * 38)

In [ ]:
# ── Decision Boundary: Predicted Probability Distribution ────────────────────
# Shows how confident the model is — well-separated peaks = good model.
# A bad model would show overlapping distributions near 0.5.
proba   = model.predict_proba(X_test).flatten()
y_test_ = y_test.flatten()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(proba[y_test_==0], bins=30, alpha=0.6, color="#E74C3C", label="Malignant (0)")
ax.hist(proba[y_test_==1], bins=30, alpha=0.6, color="#2ECC71", label="Benign (1)")
ax.axvline(x=config["training"]["threshold"], color="black",
           linestyle="--", linewidth=1.5, label=f"Threshold = {config['training']['threshold']}")
ax.set_xlabel("Predicted Probability", fontsize=12)
ax.set_ylabel("Count",                 fontsize=12)
ax.set_title("Predicted Probability Distribution by Class", fontsize=14)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../assets/probability_distribution_logistic.png", dpi=150)
plt.show()
print("Saved → assets/probability_distribution_logistic.png")

In [ ]:
# ── Top Feature Weights ───────────────────────────────────────────────────────
# Which features does the model rely on most?
# Larger absolute weight = stronger influence on prediction.
feature_names = data.feature_names
weights_flat  = model.weights.flatten()
sorted_idx    = np.argsort(np.abs(weights_flat))[::-1][:10]   # top 10

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#2ECC71" if w > 0 else "#E74C3C" for w in weights_flat[sorted_idx]]
ax.barh(feature_names[sorted_idx][::-1], weights_flat[sorted_idx][::-1], color=colors[::-1])
ax.set_xlabel("Weight Value", fontsize=12)
ax.set_title("Top 10 Feature Weights (Green=Benign, Red=Malignant)", fontsize=13)
ax.axvline(x=0, color="black", linewidth=0.8)
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("../assets/feature_weights_logistic.png", dpi=150)
plt.show()
print("Saved → assets/feature_weights_logistic.png")

## Results Summary

| Metric | Value |
|--------|-------|
| Accuracy  | *see output above* |
| Precision | *see output above* |
| Recall    | *see output above* |
| F1 Score  | *see output above* |

**Key Takeaways:**
- **Sigmoid** squashes any real number into $(0, 1)$ — interpretable as probability  
- **L2 regularization** shrinks weights toward zero, preventing overfitting  
- **Probability distribution plot**: well-separated peaks near 0 and 1 = confident model  
- **Recall matters more than precision** for cancer detection —  
  a false negative (missed cancer) is far worse than a false positive